# Manuscript

Render it to markdown and pdf

:warning: TODO: Use a different writer for each medium type

In [ ]:
# Can Render to a PDF using pandoc
!sudo apt -y install pandoc
%pip install markdown2 pypandoc

# If using pdflatex
!sudo apt -y install texlive texlive-latex-extra

# Convert to Markdown and LaTeX

## Load the manuscript

In [10]:
import settings
from model import Story

story = Story.load_from_directory(settings.STORY_DIR + "/step_14")

In [11]:
def story_to_markdown(story):
    """
    Converts a Story and its associated StoryDialog to a Markdown manuscript, formatted like a novel with chapter pages.
    """
    # Start with the title page
    markdown_content = f"# {story.title}\n\n"
    
    # Loop over each act and scene to create the story content
    for act_index, (act, act_dialog) in enumerate(zip(story.acts, story.get_story_dialogue().act_dialogues)):
        # Add a chapter title page for each act
        markdown_content += f"\n\n# Chapter {act_index + 1}\n\n"
        markdown_content += f"_{act.description}_\n\n"  # Italicize the chapter description for a thematic touch
        markdown_content += "<div style='page-break-after: always;'></div>\n\n"  # Page break for chapter separation
        
        # Loop over each scene in the act
        for scene_index, (scene, scene_dialog) in enumerate(zip(act.scenes, act_dialog.scene_dialogues)):
            markdown_content += f"\n\n---\n\n"  # Separator for scenes
            
            # Optional: Add scene description for context
            markdown_content += f"_{scene.description}_\n\n" if scene.description else ""
            
            # Add each dialogue line for the scene
            for dialogue_line in scene_dialog.dialogues:
                # Dialogue formatted as prose
                markdown_content += f"{dialogue_line.character_nickname}: {dialogue_line.line}\n\n"
    
    return markdown_content


In [12]:
# Render the markdown
markdown_content = story_to_markdown(story)

In [13]:
# Save it
import os
os.makedirs(f"{settings.STORY_DIR}/step_14", exist_ok=True)
manuscript_md_path = f"{settings.STORY_DIR}/step_14/manuscript.md"

# Save to a Markdown file
with open(manuscript_md_path, "w") as file:
    file.write(markdown_content)

In [14]:
import markdown2
import pypandoc

def markdown_to_latex(markdown_text):
    """
    Convert markdown text to LaTeX using markdown2 and pypandoc.
    """
    # Convert markdown to HTML first
    html_text = markdown2.markdown(markdown_text)
    # Convert HTML to LaTeX
    latex_text = pypandoc.convert_text(html_text, 'latex', format='html')
    return latex_text

In [15]:
def story_to_latex_graphic_novel(story, scene_image_dir):
    """
    Converts a Story and its associated StoryDialog to a LaTeX manuscript formatted as a graphic novel,
    where each scene is treated as a chapter and may include an image.
    """

    # Start with document preamble
    latex_content = r"""
    \documentclass[12pt]{report}  % Using 'report' class to avoid blank pages between chapters
    \usepackage{setspace}
    \usepackage{titlesec}
    \usepackage{graphicx}
    \usepackage{url}
    \titleformat{\chapter}[display]
      {\normalfont\huge\bfseries}{\chaptername\ \thechapter}{20pt}{\Huge}
    \usepackage{fancyhdr}
    \pagestyle{fancy}
    \fancyhf{}
    \rhead{\thepage}
    \begin{document}
    \onehalfspacing
    \pagenumbering{gobble} % Suppress page numbers until Chapter 1
    """

    # Title page with title image if available, without a page break
    # title_image_path = os.path.join(scene_image_dir, "title.png")
    title_image_path = os.path.join(scene_image_dir, "cover.png")
    latex_content += r"\begin{center}\n"
    
    # Include the title image if available
    if os.path.exists(title_image_path):
        latex_content += f"\\includegraphics[width=0.8\\textwidth]{{{title_image_path}}}\n\\vspace{{1cm}}\n"
    
    # Add the title text below the image
    latex_content += r"""
    {\Huge \textbf{""" + story.title + r"""}} \\[2cm]
    \end{center}
    """

    # Start page numbering with Chapter 1
    latex_content += r"""
    \newpage
    \pagenumbering{arabic} % Start page numbering
    """

    # Loop over each scene to create the story content
    chapter_counter = 1
    for act, act_dialogue in zip(story.acts, story.get_story_dialogue().act_dialogues):
        for scene, scene_dialogue in zip(act.scenes, act_dialogue.scene_dialogues):
            # Each scene is a chapter
            latex_content += f"\\chapter*{{Chapter {chapter_counter}: {scene.title}}}\n"
            latex_content += f"\\addcontentsline{{toc}}{{chapter}}{{Chapter {chapter_counter}: {scene.title}}}\n"
            chapter_counter += 1

            # Optional scene description in italics
            if scene.description:
                latex_content += f"\\textit{{{scene.description}}}\n\n"
            
            # Include scene image if it exists
            scene_image_path = os.path.join(scene_image_dir, f"{scene.scene_id}.live.png")
            if os.path.exists(scene_image_path):
                latex_content += f"\\begin{{center}}\n\\includegraphics[width=0.8\\textwidth]{{{scene_image_path}}}\n\\end{{center}}\n\n"

            # Use the content field for the full scene dialogue and actions, converting from markdown to latex
            if scene_dialogue.content:
                latex_scene_content = markdown_to_latex(scene_dialogue.content.strip())
                latex_content += f"{latex_scene_content}\n\n"

    # Add "Made with Plotomatic" statement at the end
    latex_content += r"""
    \newpage
    \vfill
    \begin{center}
    \textit{Made with \textbf{Plotomatic}.} \\
    For more information, visit: \url{https://github.com/mattwilliamson/Plotomatic}
    \end{center}
    """

    # End the document
    latex_content += "\\end{document}"
    
    return latex_content


In [16]:
# latex_content = story_to_latex_novel(story)
latex_content = story_to_latex_graphic_novel(story, settings.STORY_DIR + "/step_6/scenes")
latex_content

'\n    \\documentclass[12pt]{report}  % Using \'report\' class to avoid blank pages between chapters\n    \\usepackage{setspace}\n    \\usepackage{titlesec}\n    \\usepackage{graphicx}\n    \\usepackage{url}\n    \\titleformat{\\chapter}[display]\n      {\\normalfont\\huge\\bfseries}{\\chaptername\\ \\thechapter}{20pt}{\\Huge}\n    \\usepackage{fancyhdr}\n    \\pagestyle{fancy}\n    \\fancyhf{}\n    \\rhead{\\thepage}\n    \\begin{document}\n    \\onehalfspacing\n    \\pagenumbering{gobble} % Suppress page numbers until Chapter 1\n    \\begin{center}\\n\\includegraphics[width=0.8\\textwidth]{stories/my_story/step_6/scenes/cover.png}\n\\vspace{1cm}\n\n    {\\Huge \\textbf{Beneath the Surface of Deceit}} \\\\[2cm]\n    \\end{center}\n    \n    \\newpage\n    \\pagenumbering{arabic} % Start page numbering\n    \\chapter*{Chapter 1: The Perfect Life}\n\\addcontentsline{toc}{chapter}{Chapter 1: The Perfect Life}\n\\textit{Emma\'s daily routine, showcasing her successful psychology practice 

In [17]:
from IPython.display import display, Markdown

step_path = f"{settings.STORY_DIR}/step_14"
manuscript_json_path = f"{step_path}/manuscript.json"
manuscript_latex_path = f"{step_path}/manuscript.tex"
manuscript_pdf_path = f"{step_path}/manuscript.pdf"

# Save to a LaTeX file
with open(manuscript_latex_path, "w") as file:
    file.write(latex_content)

# !pandoc {manuscript_latex_path} -o {manuscript_pdf_path}
!ls -lah {step_path}
!rm manuscript.log
!pdflatex -interaction=nonstopmode -output-directory={step_path} {manuscript_latex_path}

total 2.1M
drwxrwxr-x 2 matt matt 4.0K Nov 16 19:03 .
drwxrwxr-x 8 matt matt 4.0K Nov 16 16:22 ..
-rw-rw-r-- 1 matt matt  760 Nov 18 10:34 manuscript.aux
-rw-rw-r-- 1 matt matt  52K Nov 18 10:34 manuscript.log
-rw-rw-r-- 1 matt matt  910 Nov 18 10:36 manuscript.md
-rw-rw-r-- 1 matt matt 1.6M Nov 18 10:34 manuscript.pdf
-rw-rw-r-- 1 matt matt 198K Nov 18 10:36 manuscript.tex
-rw-rw-r-- 1 matt matt 199K Nov 18 10:36 story_dialog.json
-rw-rw-r-- 1 matt matt  59K Nov 18 10:15 story.json


rm: cannot remove 'manuscript.log': No such file or directory
This is pdfTeX, Version 3.141592653-2.6-1.40.22 (TeX Live 2022/dev/Debian) (preloaded format=pdflatex)
 restricted \write18 enabled.
entering extended mode
(./stories/my_story/step_14/manuscript.tex
LaTeX2e <2021-11-15> patch level 1
L3 programming layer <2022-01-21>
(/usr/share/texlive/texmf-dist/tex/latex/base/report.cls
Document Class: report 2021/10/04 v1.4n Standard LaTeX document class
(/usr/share/texlive/texmf-dist/tex/latex/base/size12.clo))
(/usr/share/texlive/texmf-dist/tex/latex/setspace/setspace.sty)
(/usr/share/texlive/texmf-dist/tex/latex/titlesec/titlesec.sty)
(/usr/share/texlive/texmf-dist/tex/latex/graphics/graphicx.sty
(/usr/share/texlive/texmf-dist/tex/latex/graphics/keyval.sty)
(/usr/share/texlive/texmf-dist/tex/latex/graphics/graphics.sty
(/usr/share/texlive/texmf-dist/tex/latex/graphics/trig.sty)
(/usr/share/texlive/texmf-dist/tex/latex/graphics-cfg/graphics.cfg)
(/usr/share/texlive/texmf-dist/tex/latex

In [18]:
display(Markdown(f"# :open_book: [Download the manuscript PDF]({manuscript_pdf_path})"))

# :open_book: [Download the manuscript PDF](stories/my_story/step_14/manuscript.pdf)

# Generate Dialog (for screenplays)
